# Free Qwen text-to-SQL: Phases 0–3
Use the free T4 runtime and keep this notebook private. The experiment is the same frozen NF4/FP16 profile as the Kaggle notebook. Runtime files are temporary. The final export cell downloads a private checkpoint archive; run it before leaving the session. No Google Drive mount, paid runtime or API billing is enabled. This notebook has not yet produced a model benchmark.


In [ ]:
import os, pathlib, shutil, zipfile, io, base64, hashlib, subprocess, sys
PROJECT = pathlib.Path('/content/sqlrl_temp/sql-execution-grpo')
PERSIST = pathlib.Path('/content/sqlrl_free_run')
# Upload/extract a prior private output and enter its sqlrl_free_run path.
RESUME_FROM = ''
RUN_LOCAL = True
RUN_GEMINI = False
BILLING_DISABLED_CONFIRMED = False
MAX_REQUESTS = 20
if RESUME_FROM and not PERSIST.exists():
    shutil.copytree(RESUME_FROM, PERSIST)
PERSIST.mkdir(parents=True, exist_ok=True)
PROJECT.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'a3ce146c5b86120c21a061d124e3e8b4abc5ab37de29bdb9870ad9c13c955db7'
snapshot = PERSIST / 'source_snapshot.zip'
if snapshot.exists(): assert snapshot.read_bytes() == payload, 'Resume requires the same source snapshot'
else: snapshot.write_bytes(payload)
# Only results and processed records are included in the output archive.
for relative, target in [('results', PERSIST/'results'), ('data/processed', PERSIST/'processed')]:
    target.mkdir(exist_ok=True)
    link = PROJECT / relative
    if not link.exists():
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(target, target_is_directory=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        path = PROJECT / name
        if path.exists():
            assert path.read_bytes() == archive.read(name), f'Changed source or metadata: {name}'
        else:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive.read(name))
os.chdir(PROJECT)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['SQLRL_FREE_GPU'] = '1'  # Use only on your confirmed free allocation.
os.environ['HF_HOME'] = '/content/sqlrl_temp/sqlrl_hf_cache'
print('Source SHA256:', hashlib.sha256(payload).hexdigest())


In [ ]:
# Official PyPI and PyTorch CUDA 12.6 wheels; no cloud billing setup.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.13.7'], check=True)
ENV_DIR = '/content/sqlrl_temp/sqlrl_venv'
if not pathlib.Path(ENV_DIR, 'bin/python').exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', ENV_DIR], check=True)
PY = str(pathlib.Path(ENV_DIR, 'bin/python'))
subprocess.run(['uv', 'pip', 'sync', '--python', PY, 'requirements-free.lock', '--extra-index-url', 'https://download.pytorch.org/whl/cu126', '--index-strategy', 'unsafe-best-match'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-e', '.'], check=True)
subprocess.run([PY, '-m', 'pytest'], check=True)
if RUN_LOCAL:
    subprocess.run([PY, '-c', "import torch; assert torch.cuda.is_available(), 'Select a free GPU in Colab Runtime > Change runtime type'; p=torch.cuda.get_device_properties(0); print(torch.__version__,p.name,p.total_memory,torch.cuda.get_arch_list()); x=torch.ones(8,device='cuda'); print((x+x).sum().item())"], check=True)


In [ ]:
# Downloads only TRAIN. Temporary DBs/archives are not persistent output.
scratch = pathlib.Path('/content/sqlrl_temp/sqlrl_data')
scratch.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage(scratch).free > 70 * 1024**3, 'Need scratch space for 31 GB DBs, archives and model cache'
for name in ['raw', 'databases']:
    target = scratch / name
    target.mkdir(exist_ok=True)
    link = PROJECT / 'data' / name
    if not link.exists():
        link.parent.mkdir(parents=True, exist_ok=True)
        if link.is_symlink(): link.unlink()
        link.symlink_to(target, target_is_directory=True)
subprocess.run([PY, 'scripts/prepare_free_data.py'], check=True)


In [ ]:
# Zero-shot, few-shot, QLoRA SFT and SFT evaluation. Resume completed stages.
if RUN_LOCAL:
    subprocess.run([PY, 'scripts/run_free.py', '--stage', 'local', '--max-seconds', '10800'], check=True)


In [ ]:
# Optional separate CPU/API session. Never paste your key into this cell.
if RUN_GEMINI:
    assert BILLING_DISABLED_CONFIRMED, 'Verify the API project is Free tier with billing disabled'
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['SQLRL_GEMINI_FREE_PROJECT'] = '1'
    try:
        subprocess.run([PY, 'scripts/run_free.py', '--stage', 'gemini', '--max-requests', str(MAX_REQUESTS)], check=True)
    finally:
        os.environ.pop('GEMINI_API_KEY', None)


In [ ]:
# Publish no accuracy comparison until all four full-cohort evaluations exist.
needed = [PROJECT/'results/free_runs'/f'{name}_evaluation/summary.json' for name in ['base_zero','base_few','sft_eval','frontier']]
if all(path.exists() for path in needed):
    subprocess.run([PY, 'scripts/run_free.py', '--stage', 'report'], check=True)
    print((PROJECT/'results/free_runs/comparison.md').read_text())
else:
    print('Experiment pending. Preserve this PRIVATE notebook output and resume unfinished stages.')


In [ ]:
# Run before leaving this temporary session, including after a paused stage.
from google.colab import files
archive_path = shutil.make_archive('/content/sqlrl_free_run_checkpoint', 'zip', root_dir=PERSIST.parent, base_dir=PERSIST.name)
print('Private results/checkpoints archive:', archive_path)
files.download(archive_path)
